In [1]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader

In [2]:
ds_gfs = xr.open_dataset('GFS_UPDATED_V1.nc')
ds_sla = xr.open_dataset('SLA_UPDATED.nc')

In [3]:
variables = [
    "sla",
    "TMP_surface",
    "UGRD_10maboveground",
    "VGRD_10maboveground",
    "PRATE_surface",
    "DSWRF_surface",
    "USWRF_surface",
    "DLWRF_surface",
    "ULWRF_surface",
    "SPFH_2maboveground"
]

In [4]:
ds = xr.merge([
    ds_sla[["sla"]],
    ds_gfs[variables[1:]]
])

In [5]:
lead = 1
target = ds['sla'].shift(time=-lead)

In [6]:
ds = ds.isel(time=slice(None, -lead))
target = target.isel(time=slice(None, -lead))

In [7]:
train_ds = ds.sel(time=slice(None, "2023-12-31"))

test_ds = ds.sel(time=slice("2024-01-01", None))

train_target = target.sel(time=slice(None, "2023-12-31"))

test_target = target.sel(time=slice("2024-01-01", None))

In [8]:
means = {}
stds = {}

for var in variables:

    means[var] = train_ds[var].mean(skipna=True)

    stds[var] = train_ds[var].std(skipna=True)

    print(
        f"{var:25}",
        f"mean={float(means[var]):10.4f}",
        f"std={float(stds[var]):10.4f}"
    )

sla                       mean=    0.0919 std=    0.0905
TMP_surface               mean=  301.4708 std=    5.0400
UGRD_10maboveground       mean=    1.1343 std=    4.0516
VGRD_10maboveground       mean=    0.2166 std=    3.7267
PRATE_surface             mean=    0.0000 std=    0.0001
DSWRF_surface             mean=  250.9850 std=   54.7321
USWRF_surface             mean=   27.6067 std=   21.4303
DLWRF_surface             mean=  395.3973 std=   39.2619
ULWRF_surface             mean=  467.5513 std=   30.8340
SPFH_2maboveground        mean=    0.0149 std=    0.0049


In [9]:
def normalize_dataset(ds, means, stds, variables):

    ds_norm = ds.copy()

    for var in variables:
        ds_norm[var] = (ds[var] - means[var]) / stds[var]

    return ds_norm

In [10]:
train_ds.values

<bound method Mapping.values of <xarray.Dataset> Size: 2GB
Dimensions:              (time: 3286, latitude: 116, longitude: 160)
Coordinates:
  * time                 (time) datetime64[ns] 26kB 2015-01-01 ... 2023-12-31
  * latitude             (latitude) float32 464B 0.125 0.375 ... 28.62 28.88
  * longitude            (longitude) float32 640B 50.12 50.38 ... 89.62 89.88
Data variables:
    sla                  (time, latitude, longitude) float32 244MB -0.0456 .....
    TMP_surface          (time, latitude, longitude) float32 244MB 300.9 ... ...
    UGRD_10maboveground  (time, latitude, longitude) float32 244MB -3.028 ......
    VGRD_10maboveground  (time, latitude, longitude) float32 244MB -6.598 ......
    PRATE_surface        (time, latitude, longitude) float32 244MB 1.719e-06 ...
    DSWRF_surface        (time, latitude, longitude) float32 244MB 292.7 ... ...
    USWRF_surface        (time, latitude, longitude) float32 244MB 19.34 ... ...
    DLWRF_surface        (time, latitude, l

In [11]:
train_ds = normalize_dataset(train_ds, means, stds, variables)

test_ds = normalize_dataset(test_ds, means, stds, variables)

In [12]:
train_target = (train_target - means["sla"]) / stds["sla"]

test_target = (test_target - means["sla"]) / stds["sla"]

In [13]:
ocean_mask = ~ds_sla.sla.isel(time=0).isnull()

In [14]:
train_ds_norm = normalize_dataset(train_ds, means, stds, variables)
test_ds_norm = normalize_dataset(test_ds, means, stds, variables)

train_target_norm = (train_target - means["sla"]) / stds["sla"]
test_target_norm = (test_target - means["sla"]) / stds["sla"]

In [15]:
print(train_ds["sla"].mean(skipna=True).item())
print(train_ds["sla"].std(skipna=True).item())

-7.25094650988467e-08
0.9999999403953552


In [16]:
history = 14
lead = 1

In [17]:
n_samples = len(train_ds_norm.time) - history - lead + 1

print(n_samples)

3272


In [18]:
idx = 0

In [19]:
x = train_ds_norm.isel(
    time=slice(idx, idx + history)
)

In [20]:
y = train_target_norm.isel(
    time=idx + history - 1
)

In [21]:
class SLADataset(Dataset):

    def __init__(self, ds, target, variables, history):

        self.ds = ds
        self.target = target
        self.variables = variables
        self.history = history

        self.n_samples = len(ds.time) - history + 1

    def __len__(self):

        return self.n_samples

    def __getitem__(self, idx):

        x = self.ds.isel(
            time=slice(idx, idx + self.history)
        )

        y = self.target.isel(
            time=idx + self.history - 1
        )

        x = np.stack(
            [x[var].values for var in self.variables],
            axis=1
        )

        y = y.values[np.newaxis, :, :]

        x = np.nan_to_num(x, nan=0.0)
        y = np.nan_to_num(y, nan=0.0)

        x = torch.tensor(x, dtype=torch.float32)
        y = torch.tensor(y, dtype=torch.float32)

        return x, y

In [22]:
train_dataset = SLADataset(
    train_ds_norm,
    train_target_norm,
    variables,
    history
)

test_dataset = SLADataset(
    test_ds_norm,
    test_target_norm,
    variables,
    history
)

In [23]:
len(train_dataset)

3273

In [24]:
x, y = train_dataset[0]

print(x.shape)
print(y.shape)

torch.Size([14, 10, 116, 160])
torch.Size([1, 116, 160])


In [25]:
print(torch.isnan(x).sum())
print(torch.isnan(y).sum())

tensor(0)
tensor(0)


In [26]:
train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

In [31]:
test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

In [32]:
x, y = next(iter(train_loader))

print(x.shape)
print(y.shape)

torch.Size([8, 14, 10, 116, 160])
torch.Size([8, 1, 116, 160])


In [ ]:
conv = nn.Conv2d(
    in_channels=10,
    out_channels=32,
    kernel_size=3,
    padding=1
)

In [ ]:
x = torch.randn(
    8,
    10,
    116,
    160
)

In [ ]:
y = conv(x)

print(y.shape)

torch.Size([8, 32, 116, 160])


In [33]:
class CNNEncoder(nn.Module):

    def __init__(self,
                 in_channels=10,
                 hidden_channels=32):

        super().__init__()

        self.encoder = nn.Sequential(

            nn.Conv2d(
                in_channels=in_channels,
                out_channels=16,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(),

            nn.Conv2d(
                in_channels=16,
                out_channels=hidden_channels,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU()

        )

    def forward(self, x):

        batch, time, channels, height, width = x.shape

        # Merge batch and time
        x = x.reshape(
            batch * time,
            channels,
            height,
            width
        )

        # CNN
        x = self.encoder(x)

        # Recover sequence dimension
        x = x.reshape(
            batch,
            time,
            32,
            height,
            width
        )

        return x

In [34]:
encoder = CNNEncoder()

y = encoder(x)

print(y.shape)

torch.Size([8, 14, 32, 116, 160])


In [35]:
class ConvLSTMCell(nn.Module):

    def __init__(self,
                 input_channels=32,
                 hidden_channels=32):

        super().__init__()

        self.hidden_channels = hidden_channels

        self.conv = nn.Conv2d(
            in_channels=input_channels + hidden_channels,
            out_channels=4 * hidden_channels,
            kernel_size=3,
            padding=1
        )